In [12]:
import yfinance as yf

days = 100 
ticker = "SBUX"
interval = "1h"

sbux = yf.download(
    tickers    = ticker,
    period     = f"{days}d",
    interval   = interval,
    auto_adjust = True,
    prepost    = False
)

print("Downloaded shape:", sbux.shape)
print(sbux.tail(5))

[*********************100%***********************]  1 of 1 completed

Downloaded shape: (683, 5)
Price                          Close       High        Low       Open  Volume
Ticker                          SBUX       SBUX       SBUX       SBUX    SBUX
Datetime                                                                     
2026-02-27 16:30:00+00:00  98.160004  98.425003  97.750000  97.894997       0
2026-02-27 17:30:00+00:00  98.059998  98.339996  97.839996  98.165001  292947
2026-02-27 18:30:00+00:00  97.430000  98.080002  97.415001  98.059998  213576
2026-02-27 19:30:00+00:00  97.949997  98.010002  97.360001  97.425003  299648
2026-02-27 20:30:00+00:00  98.029999  98.209999  97.860001  97.949997  757837


In [13]:
# 1 - data_fetcher
from yfinance import tickers
from data_fetcher import clean_yfinance_data 

clean_df = clean_yfinance_data(sbux)
clean_df.to_parquet(f"data/sbux_1h_clean.parquet", compression="gzip")

print("\nData is cleaned and saved:")
print(clean_df.head())
print(f"\nFinal shape: {clean_df.shape}")

Cleaned shape: (683, 5)
Date range: 2025-10-06 09:30:00-04:00 → 2026-02-27 15:30:00-05:00
Timezone: America/New_York

Data is cleaned and saved:
Price                           Open       High        Low      Close   Volume
Datetime                                                                      
2025-10-06 09:30:00-04:00  86.279999  86.290001  82.779999  83.014999  3278488
2025-10-06 10:30:00-04:00  83.010002  83.330002  82.959999  83.044998  1700777
2025-10-06 11:30:00-04:00  83.040001  83.150002  82.491096  82.669998  1869634
2025-10-06 12:30:00-04:00  82.669998  82.849998  82.550003  82.614998  1232110
2025-10-06 13:30:00-04:00  82.629997  83.080002  82.589996  82.629997  1492652

Final shape: (683, 5)


In [14]:
# 2 - regime_detection
from regime_detector import load_clean_data, detect_regime #, save_regime_plot

df = load_clean_data()
df_regime = detect_regime(df)
#save_regime_plot(df_regime)

print("\nLast 8 bars (now with real ADX values):")
print(df_regime[['Close', 'ADX', 'ATR', 'regime']].tail(8))

Loaded clean SBUX 1H data: 683 bars
Date range: 2025-10-06 09:30:00-04:00 → 2026-02-27 15:30:00-05:00

SBUX 1H Regime Breakdown (valid bars only):
Trending (→ Trend Following):   47.5%
Range-bound (→ Mean Reversion): 52.5%

Last 8 bars (now with real ADX values):
Price                          Close        ADX       ATR       regime
Datetime                                                              
2026-02-26 15:30:00-05:00  98.110001  31.522156  0.736489     trending
2026-02-27 09:30:00-05:00  98.129997  29.459237  0.846025     trending
2026-02-27 10:30:00-05:00  97.885002  27.543670  0.832738     trending
2026-02-27 11:30:00-05:00  98.160004  25.680706  0.821471     trending
2026-02-27 12:30:00-05:00  98.059998  23.950811  0.798509  range_bound
2026-02-27 13:30:00-05:00  97.430000  22.908458  0.788973  range_bound
2026-02-27 14:30:00-05:00  97.949997  22.012062  0.779046  range_bound
2026-02-27 15:30:00-05:00  98.029999  20.852909  0.748400  range_bound


In [15]:
from regime_detector import load_clean_data, detect_regime #,save_regime_plot
import pandas as pd
from pathlib import Path

df = load_clean_data()
df_regime = detect_regime(df)
#save_regime_plot(df_regime)

df_regime.to_parquet("data/sbux_1h_with_regime.parquet", compression="gzip")
print("Regime-enhanced data SAVED successfully to data/sbux_1h_with_regime.parquet")
print(f"File size: {Path('data/sbux_1h_with_regime.parquet').stat().st_size / 1024:.1f} KB")

Loaded clean SBUX 1H data: 683 bars
Date range: 2025-10-06 09:30:00-04:00 → 2026-02-27 15:30:00-05:00

SBUX 1H Regime Breakdown (valid bars only):
Trending (→ Trend Following):   47.5%
Range-bound (→ Mean Reversion): 52.5%
Regime-enhanced data SAVED successfully to data/sbux_1h_with_regime.parquet
File size: 35.9 KB


In [16]:
# 3 - mean_reversion
from mean_reversion import load_regime_data, mean_reversion_logic

df = load_regime_data()
df_mr = mean_reversion_logic(df)

print("\nSample of completed MR trades (first 8 entries with exits):")
completed = df_mr[(df_mr['mr_signal'] == 1) | (df_mr['mr_exit'] == 1)]
print(completed[['Close', 'zscore', 'regime', 'mr_signal', 'mr_exit']].head(20))


df_mr.to_parquet("data/sbux_1h_mr_signals.parquet", compression="gzip")
print("\nMean Reversion signals saved to: data/sbux_1h_mr_signals.parquet")

Loaded regime-enhanced SBUX 1H data: 683 bars

Mean Reversion Strategy (range-bound only, SBUX 1H):
   Long entries: 8
   Exits (z-stop or mean cross): 8
   Open trades at end: 0

Sample of completed MR trades (first 8 entries with exits):
Price                          Close    zscore       regime  mr_signal  \
Datetime                                                                 
2025-10-28 15:30:00-04:00  85.400002 -2.315853  range_bound          1   
2025-11-05 10:30:00-05:00  80.349998  0.135007     trending          0   
2025-11-18 09:30:00-05:00  82.614998 -2.068298  range_bound          1   
2025-11-19 10:30:00-05:00  84.139999  0.127047  range_bound          0   
2025-12-08 13:30:00-05:00  83.449997 -2.068110  range_bound          1   
2025-12-10 14:30:00-05:00  83.620003  0.791458     trending          0   
2025-12-31 09:30:00-05:00  84.309998 -2.185375  range_bound          1   
2026-01-02 13:30:00-05:00  84.889999  0.237683  range_bound          0   
2026-01-27 09:30:00-

In [17]:
# 4 - trend_following
from trend_following import load_regime_data, trend_following_logic

df = load_regime_data()
df_tf = trend_following_logic(df)

print("\nSample of completed Trend Following trades (first 10 entries + exits):")
completed = df_tf[(df_tf['tf_signal'] == 1) | (df_tf['tf_exit'] == 1)]
print(completed[['Close', 'ma_fast', 'ma_slow', 'regime', 'tf_signal', 'tf_exit']].head(25))

df_tf.to_parquet("data/sbux_1h_tf_signals.parquet", compression="gzip")
print("\nTrend Following signals saved to: data/sbux_1h_tf_signals.parquet")

Loaded regime-enhanced SBUX 1H data: 683 bars

Trend Following Strategy (trending regimes only, SBUX 1H):
   Long entries: 7
   Exits (ATR trailing stop): 6
   Open trades at end: 1

Sample of completed Trend Following trades (first 10 entries + exits):
Price                          Close    ma_fast    ma_slow       regime  \
Datetime                                                                  
2025-10-15 09:30:00-04:00  82.760002  80.872379  80.630178     trending   
2025-10-28 15:30:00-04:00  85.400002  86.699000  85.925454  range_bound   
2025-11-12 09:30:00-05:00  88.440002  86.213271  83.020004     trending   
2025-11-14 10:30:00-05:00  84.550003  86.446809  85.082970     trending   
2025-12-12 15:30:00-05:00  85.349998  85.210511  84.383392     trending   
2025-12-22 11:30:00-05:00  87.294998  88.472850  86.528725     trending   
2026-01-06 09:30:00-05:00  89.209999  86.334500  85.061864     trending   
2026-01-07 09:30:00-05:00  87.220001  88.678500  85.808264     trending

In [18]:
# 5 - risk_manager
from risk_manager import RiskManager

rm = RiskManager(initial_capital=100_000)
print(f"Can trade? {rm.can_trade()}")

# Simulate one trade
size = rm.calculate_position_size(atr=0.80, price=98.0)
print(f"Position size for current ATR: {size} shares (risks ~$500)")

# Testing for risk manager with small profit
rm.update_equity(98_500)  
print(f"Equity now: ${rm.current_equity:,.0f}")

RiskManager initialized: $100,000 capital, 0.5% per-trade risk
Can trade? True
Position size for current ATR: 416 shares (risks ~$500)
Equity now: $98,500


In [19]:
# 6 - backtester
from backtester import load_all_data, run_backtest, calculate_metrics

df = load_all_data()
df_bt = run_backtest(df)

# Split & metrics
split_idx = int(len(df_bt) * 0.7)
print("\n=== IN-SAMPLE (70%) ===")
calculate_metrics(df_bt.iloc[:split_idx])
print("\n=== OUT-OF-SAMPLE (30%) ===")
calculate_metrics(df_bt.iloc[split_idx:])

# Quick equity curve preview
print("\nLast 5 equity values:")
print(df_bt[['Close', 'equity', 'position']].tail())

df_bt.to_parquet("data/sbux_1h_backtest_results.parquet", compression="gzip")
print("Full backtest saved → data/sbux_1h_backtest_results.parquet")

Merged full dataset: 683 bars
RiskManager initialized: $100,000 capital, 0.5% per-trade risk

=== IN-SAMPLE (70%) ===

Full Period Metrics (SBUX 1H)
   Total Return:           3.0%
   Sharpe (daily)         0.92   ← annualized using daily returns
   Max Drawdown:          -0.8%
   Win Rate:              80.0%
   Profit Factor:         4.44
   Number of trades:         5

=== OUT-OF-SAMPLE (30%) ===

Full Period Metrics (SBUX 1H)
   Total Return:           2.8%
   Sharpe (daily)         1.97   ← annualized using daily returns
   Max Drawdown:          -0.3%
   Win Rate:              66.7%
   Profit Factor:        10.60
   Number of trades:         3

Last 5 equity values:
Price                          Close         equity  position
Datetime                                                     
2026-02-27 11:30:00-05:00  98.160004  105847.877945       366
2026-02-27 12:30:00-05:00  98.059998  105847.877945       366
2026-02-27 13:30:00-05:00  97.430000  105847.877945       366
2026-02-27

In [20]:
from optimizer import optimize_parameters

best_params = optimize_parameters()
print("\nOptimization complete")

Merged full dataset: 683 bars
Starting limited in-sample optimization (5 combos only — PJ Sutherland style)...


Mean Reversion Strategy (range-bound only, SBUX 1H):
   Long entries: 6
   Exits (z-stop or mean cross): 6
   Open trades at end: 0

Trend Following Strategy (trending regimes only, SBUX 1H):
   Long entries: 5
   Exits (ATR trailing stop): 5
   Open trades at end: 0
RiskManager initialized: $100,000 capital, 0.5% per-trade risk

IS - z=1.8, MA=8/40, ATRx2.5 Metrics (SBUX 1H)
   Total Return:           4.6%
   Sharpe (daily)         1.58   ← annualized using daily returns
   Max Drawdown:          -0.8%
   Win Rate:              71.4%
   Profit Factor:         5.31
   Number of trades:         7

Mean Reversion Strategy (range-bound only, SBUX 1H):
   Long entries: 4
   Exits (z-stop or mean cross): 4
   Open trades at end: 0

Trend Following Strategy (trending regimes only, SBUX 1H):
   Long entries: 4
   Exits (ATR trailing stop): 4
   Open trades at end: 0
RiskManager ini

In [21]:
from mean_reversion import mean_reversion_logic
from trend_following import trend_following_logic
from backtester import load_all_data, run_backtest, calculate_metrics

df = load_all_data()

# ─── Remove any old/existing signal columns to avoid overlap ───
signal_cols = [
    'mr_signal', 'mr_exit', 'mr_position',
    'tf_signal', 'tf_exit', 'tf_position', 'trailing_stop'
]
df = df.drop(columns=[c for c in signal_cols if c in df.columns])

# ─── Generate fresh signals with best params ───
df_mr = mean_reversion_logic(df, z_entry=1.9)
df_tf = trend_following_logic(df, fast=9, slow=45, atr_mult=2.8)

# ─── Join the new signals ───
df_final = df.join(df_mr[['mr_signal', 'mr_exit', 'mr_position']])
df_final = df_final.join(df_tf[['tf_signal', 'tf_exit', 'tf_position', 'trailing_stop']])

# ─── Run backtest ───
df_bt_final = run_backtest(df_final)

print("\n=== FINAL FULL PERIOD (best params) ===")
calculate_metrics(df_bt_final, "Full Period - Best Params")

split_idx = int(len(df_bt_final) * 0.7)
print("\n=== OOS ONLY (last 30%) ===")
calculate_metrics(df_bt_final.iloc[split_idx:], "OOS - Best Params")

Merged full dataset: 683 bars

Mean Reversion Strategy (range-bound only, SBUX 1H):
   Long entries: 10
   Exits (z-stop or mean cross): 10
   Open trades at end: 0

Trend Following Strategy (trending regimes only, SBUX 1H):
   Long entries: 8
   Exits (ATR trailing stop): 7
   Open trades at end: 1
RiskManager initialized: $100,000 capital, 0.5% per-trade risk

=== FINAL FULL PERIOD (best params) ===

Full Period - Best Params Metrics (SBUX 1H)
   Total Return:           7.7%
   Sharpe (daily)         1.73   ← annualized using daily returns
   Max Drawdown:          -0.8%
   Win Rate:              70.0%
   Profit Factor:         6.58
   Number of trades:        10

=== OOS ONLY (last 30%) ===

OOS - Best Params Metrics (SBUX 1H)
   Total Return:           2.8%
   Sharpe (daily)         1.97   ← annualized using daily returns
   Max Drawdown:          -0.3%
   Win Rate:              66.7%
   Profit Factor:        10.58
   Number of trades:         3


{'total_return': np.float64(2.756781692940913),
 'sharpe_daily': np.float64(1.9714910176660825),
 'max_dd': np.float64(-0.27918025442974503),
 'win_rate': np.float64(66.66666666666666),
 'profit_factor': np.float64(10.582813222307603),
 'n_trades': np.int64(3)}